In [14]:
import os 
from PIL import Image 
from torchvision import transforms , datasets
from torch.utils.data import DataLoader ,random_split
import torch
from torch import nn 

In [15]:
folder=r".\data\trainingSet\trainingSet\3"
for filename in os.listdir(folder)[:5]:
    path=os.path.join(folder,filename)
    img=Image.open(path)
    print(filename,img.size,img.mode)

img_10004.jpg (28, 28) L
img_10016.jpg (28, 28) L
img_10018.jpg (28, 28) L
img_10029.jpg (28, 28) L
img_10045.jpg (28, 28) L


In [16]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [17]:
train_data = datasets.ImageFolder(
    root=r".\data\trainingSet\trainingSet",
    transform=transform
)

train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size

train_dataset, val_dataset = random_split(
    train_data,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)


In [18]:
print(train_data.classes)
print(train_data.class_to_idx)
print(len(train_data))

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
{'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}
42000


In [19]:
train_loader=DataLoader(train_data,batch_size=64,shuffle=True)

In [20]:
Images ,labels = next(iter(train_loader))
print(Images.shape)
print(labels)

torch.Size([64, 1, 28, 28])
tensor([7, 7, 1, 3, 2, 9, 4, 1, 4, 3, 5, 6, 9, 3, 5, 0, 7, 1, 8, 6, 2, 0, 3, 5,
        9, 8, 3, 6, 2, 9, 9, 4, 3, 8, 4, 8, 0, 5, 1, 1, 7, 4, 5, 1, 2, 4, 2, 9,
        2, 4, 0, 3, 0, 1, 5, 9, 9, 6, 2, 2, 9, 0, 2, 7])


In [21]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [22]:
class classficationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flat=nn.Flatten()
        self.layers=nn.Sequential(
            nn.Linear(28*28,164),nn.ReLU(),
            nn.Linear(164,164),nn.ReLU(),
            nn.Linear(164,128),nn.ReLU(),
            nn.Linear(128,128),nn.ReLU(),
            nn.Linear(128,64),nn.ReLU(),
            nn.Linear(64,10)
        )
    def forward(self,x:torch.Tensor)->torch.Tensor:
        x=self.flat(x)
        return self.layers(x)

torch.manual_seed(42)
model=classficationModel().to(device)    

In [23]:
optimizer = torch.optim.Adam(lr=.001,params=model.parameters())
loss_fn= nn.CrossEntropyLoss()

In [24]:
epochs=60
for epoch in range(epochs):
    model.train()
    running_loss = 0
    for images , labels in train_loader:
        images=images.to(device)
        labels=labels.to(device)

        optimizer.zero_grad()
        pred = model(images)
        loss=loss_fn(pred,labels)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()

    avg_loss = running_loss/len(train_loader)
    print(f"epoch {epoch+1} /{epochs} - loss : {avg_loss:.4f}")

epoch 1 /60 - loss : 0.5298
epoch 2 /60 - loss : 0.2093
epoch 3 /60 - loss : 0.1576
epoch 4 /60 - loss : 0.1331
epoch 5 /60 - loss : 0.1143
epoch 6 /60 - loss : 0.0977
epoch 7 /60 - loss : 0.0893
epoch 8 /60 - loss : 0.0774
epoch 9 /60 - loss : 0.0755
epoch 10 /60 - loss : 0.0644
epoch 11 /60 - loss : 0.0619
epoch 12 /60 - loss : 0.0612
epoch 13 /60 - loss : 0.0543
epoch 14 /60 - loss : 0.0464
epoch 15 /60 - loss : 0.0474
epoch 16 /60 - loss : 0.0455
epoch 17 /60 - loss : 0.0402
epoch 18 /60 - loss : 0.0385
epoch 19 /60 - loss : 0.0346
epoch 20 /60 - loss : 0.0343
epoch 21 /60 - loss : 0.0369
epoch 22 /60 - loss : 0.0339
epoch 23 /60 - loss : 0.0293
epoch 24 /60 - loss : 0.0316
epoch 25 /60 - loss : 0.0267
epoch 26 /60 - loss : 0.0253
epoch 27 /60 - loss : 0.0301
epoch 28 /60 - loss : 0.0246
epoch 29 /60 - loss : 0.0245
epoch 30 /60 - loss : 0.0214
epoch 31 /60 - loss : 0.0202
epoch 32 /60 - loss : 0.0260
epoch 33 /60 - loss : 0.0242
epoch 34 /60 - loss : 0.0184
epoch 35 /60 - loss : 0

In [25]:
model.eval()

correct = 0
total = 0

with torch.inference_mode():
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        pred = model(images)
        predicted_classes = torch.argmax(pred, dim=1)

        correct += (predicted_classes == labels).sum().item()
        total += labels.size(0)

train_accuracy = correct / total

print(f"Train Accuracy: {train_accuracy * 100:.2f}%")

Train Accuracy: 99.62%


In [26]:
model.eval()

correct = 0
total = 0

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

val_acc = correct / total

print(f"Validation Accuracy: {val_acc * 100:.2f}%")

Validation Accuracy: 99.57%


In [28]:
torch.save(model.state_dict(), "classification_model.pth")